# Exploración y consultas analíticas de datos XM

## Propósito

Este notebook corresponde a la etapa de **exploración, consulta y análisis** del proyecto final de Gestión de Datos Masivos.

Los datos utilizados fueron previamente obtenidos desde la API de XM, almacenados en la zona **RAW de HDFS** y posteriormente procesados mediante Apache Spark para aplicar normalización, validaciones de calidad y transformación hacia la zona **CURATED** en formato Parquet.

En este notebook no se realiza nuevamente la ingesta ni la limpieza principal de los datos. Su objetivo es consumir la información disponible en CURATED y utilizar **Apache Spark y Spark SQL** para desarrollar las consultas analíticas del proyecto.

## Alcance del análisis

El análisis se concentra en información del período comprendido entre **2020-01-01 y 2026-11-19** y busca estudiar principalmente:

1. La calidad y trazabilidad de los datos procesados.
2. El comportamiento del almacenamiento de los embalses.
3. Los aportes hídricos de los ríos frente a sus valores históricos.
4. El comportamiento de la generación hidroeléctrica bajo diferentes condiciones hídricas.
5. El comportamiento del precio de bolsa bajo diferentes condiciones hídricas.

## Flujo utilizado

```text
API XM
   ↓
HDFS RAW
   ↓
Spark: normalización y calidad
   ↓
HDFS CURATED / Parquet
   ↓
Jupyter + Spark / Spark SQL
   ↓
Consultas y análisis
   ↓
HDFS ANALYTICS
```

**HDFS** se utiliza como sistema de almacenamiento distribuido, **Apache Spark** como motor de procesamiento y **Jupyter** como entorno interactivo para ejecutar, revisar e interpretar las consultas.

Los resultados obtenidos se interpretarán como asociaciones y patrones presentes en los datos. No se asumirán relaciones causales cuando la evidencia disponible no permita demostrarlas.

In [1]:
import requests
import json

url = "https://servapibi.xm.com.co/daily"

payload = {
    "MetricId": "PorcVoluUtilDiar",
    "StartDate": "2026-10-01",
    "EndDate": "2026-10-07",
    "Entity": "Embalse"
}

respuesta = requests.post(url, json=payload, timeout=60)

print("HTTP:", respuesta.status_code)

datos = respuesta.json()

print(json.dumps(datos, indent=2, ensure_ascii=False)[:10000])

HTTP: 200
{
  "Metric": {
    "Id": "PorcVoluUtilDiar",
    "Name": "Volumen Útil  diario % por Embalse",
    "StartDate": "2026-10-01T00:00:00",
    "EndDate": "2026-10-07T00:00:00"
  },
  "Items": [
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "AGREGADO BOGOTA",
          "Value": 0.66393
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "ALTOANCHICAYA",
          "Value": 0.364
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "AMANI",
          "Value": 0.61767
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "Id": "Embalse",
          "Name": "BETANIA",
          "Value": 0.59686
        }
      ]
    },
    {
      "Date": "2026-10-01",
      "DailyEntities": [
        {
          "I

In [2]:
import pandas as pd

filas = []

for item in datos["Items"]:
    fecha = item["Date"]

    for entidad in item["DailyEntities"]:
        filas.append({
            "fecha": fecha,
            "embalse": entidad["Name"],
            "valor_original": entidad["Value"],
            "porcentaje": entidad["Value"] * 100
        })

df_embalses = pd.DataFrame(filas)

df_embalses.head(20)

,fecha,embalse,valor_original,porcentaje
0,2026-10-01,AGREGADO BOGOTA,0.66393,66.393
1,2026-10-01,ALTOANCHICAYA,0.36400,36.400
2,2026-10-01,AMANI,0.61767,61.767
3,2026-10-01,BETANIA,0.59686,59.686
4,2026-10-01,CALIMA1,0.62329,62.329
5,2026-10-01,CHUZA,0.92087,92.087
6,2026-10-01,EL QUIMBO,0.82438,82.438
7,2026-10-01,ESMERALDA,0.95922,95.922
8,2026-10-01,GUAVIO,0.88460,88.460
9,2026-10-01,ITUANGO,1.11174,111.174


In [3]:
df_embalses[
    df_embalses["fecha"] == "2026-10-01"
].sort_values("porcentaje").head(10)

,fecha,embalse,valor_original,porcentaje
16,2026-10-01,PRADO,0.35341,35.341
1,2026-10-01,ALTOANCHICAYA,0.36400,36.400
17,2026-10-01,PUNCHINA,0.40969,40.969
15,2026-10-01,PORCE III,0.48434,48.434
14,2026-10-01,PORCE II,0.53726,53.726
18,2026-10-01,RIOGRANDE2,0.54803,54.803
3,2026-10-01,BETANIA,0.59686,59.686
13,2026-10-01,PLAYAS,0.61433,61.433
23,2026-10-01,URRA1,0.61726,61.726
2,2026-10-01,AMANI,0.61767,61.767


In [4]:
menores_por_dia = (
    df_embalses
    .sort_values(["fecha", "porcentaje"])
    .groupby("fecha")
    .head(5)
    [["fecha", "embalse", "porcentaje"]]
)

menores_por_dia

,fecha,embalse,porcentaje
16,2026-10-01,PRADO,35.341
1,2026-10-01,ALTOANCHICAYA,36.400
17,2026-10-01,PUNCHINA,40.969
15,2026-10-01,PORCE III,48.434
14,2026-10-01,PORCE II,53.726
41,2026-10-02,PUNCHINA,27.980
40,2026-10-02,PRADO,36.060
25,2026-10-02,ALTOANCHICAYA,37.586
39,2026-10-02,PORCE III,51.430
38,2026-10-02,PORCE II,55.193


In [5]:
resumen_persistencia = (
    menores_por_dia
    .groupby("embalse")
    .agg(
        dias_entre_5_mas_bajos=("fecha", "count"),
        porcentaje_promedio=("porcentaje", "mean"),
        porcentaje_minimo=("porcentaje", "min"),
        porcentaje_maximo=("porcentaje", "max")
    )
    .sort_values(
        ["dias_entre_5_mas_bajos", "porcentaje_promedio"],
        ascending=[False, True]
    )
)

resumen_persistencia

,dias_entre_5_mas_bajos,porcentaje_promedio,porcentaje_minimo,porcentaje_maximo
embalse,,,,
PRADO,7,36.156000,35.341,36.628
ALTOANCHICAYA,6,39.566333,25.452,54.866
PUNCHINA,5,36.237600,13.626,55.619
PORCE II,5,50.032400,45.435,55.193
PLAYAS,4,46.361000,36.485,55.269
PORCE III,4,49.825750,41.690,57.749
ITUANGO,3,39.263667,37.967,40.586
RIOGRANDE2,1,56.285000,56.285,56.285


# Validación inicial del dataset de generación

En esta sección se realiza la lectura del dataset generacion_enriquecida almacenado en la zona CURATED de HDFS.

Este dataset fue construido previamente a partir de la información de generación (Gene) y del catálogo de recursos (ListadoRecursos). El cruce permitió incorporar atributos como nombre del recurso, tipo de generación, fuente de energía y clasificación hidráulica.

La lectura se realiza mediante Apache Spark desde Jupyter, sin copiar los datos localmente.

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Analisis_XM")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

print("Spark conectado")
print("Version:", spark.version)

Spark conectado
Version: 3.5.9


In [11]:
generacion = spark.read.parquet(
    "hdfs://namenode:9000/proyecto_xm/curated/generacion_enriquecida"
)
generacion.printSchema()
generacion.show(5, truncate=False)

root
 |-- metrica: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora: integer (nullable = true)
 |-- codigo_recurso: string (nullable = true)
 |-- generacion: double (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- nombre_recurso: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- fuente_energia: string (nullable = true)
 |-- codigo_empresa: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- despacho: string (nullable = true)
 |-- tipo_recurso: string (nullable = true)
 |-- fecha_operacion: string (nullable = true)
 |-- correspondencia_catalogo: boolean (nullable = true)
 |-- es_hidraulico: boolean (nullable = true)
 |-- year: integer (nullable = true)



+-------+----------+----+--------------+----------+--------------------------------------------------------------------------------------+------------------------+----------+--------------+--------------+---------+--------------------------+----------------+---------------+------------------------+-------------+----+
|metrica|fecha     |hora|codigo_recurso|generacion|archivo_origen                                                                        |nombre_recurso          |tipo      |fuente_energia|codigo_empresa|estado   |despacho                  |tipo_recurso    |fecha_operacion|correspondencia_catalogo|es_hidraulico|year|
+-------+----------+----+--------------+----------+--------------------------------------------------------------------------------------+------------------------+----------+--------------+--------------+---------+--------------------------+----------------+---------------+------------------------+-------------+----+
|Gene   |2026-01-01|24  |2QBW          |193

El DataFrame se registra como una vista temporal denominada generacion, con el fin de realizar consultas mediante Spark SQL.

Como primera validación se comparan:

*cantidad total de eventos disponibles

*cantidad de recursos distintos.

El objetivo es verificar que el dataset consultado desde Jupyter corresponde al mismo conjunto de datos validado durante la etapa de procesamiento y enriquecimiento.

In [12]:
generacion.createOrReplaceTempView("generacion")

In [13]:
spark.sql("""
SELECT
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT codigo_recurso) AS recursos_distintos
FROM generacion
""").show()

[Stage 6:===================================>                       (3 + 2) / 5]

+-------------+------------------+
|total_eventos|recursos_distintos|
+-------------+------------------+
|     12296013|               672|
+-------------+------------------+



### Resultado de la validación

La consulta sobre la vista temporal `generacion` retornó un total de **12.296.013 eventos de generación** correspondientes a **672 recursos distintos**.

Estos valores coinciden con los resultados obtenidos previamente durante la validación del dataset `Gene`, lo que confirma que la información almacenada en la zona **CURATED de HDFS** fue leída correctamente desde Jupyter mediante Apache Spark.

Este resultado también permite verificar que el enriquecimiento realizado con el catálogo de recursos no modificó la cantidad total de eventos de generación disponible para el análisis.

## Análisis del almacenamiento de los embalses

En esta sección se analiza la evolución del almacenamiento relativo de los embalses utilizando la métrica `PorcVoluUtilDiar`, previamente procesada y almacenada en la zona **CURATED de HDFS**.

El objetivo es responder la siguiente pregunta de negocio:

**¿Qué embalses presentan menor almacenamiento relativo respecto a su capacidad útil y cómo cambia esta condición en el tiempo?**

Antes de realizar agregaciones o comparaciones, se revisará la estructura y una muestra del dataset para verificar las variables disponibles y su granularidad temporal.

In [23]:
from IPython.display import display
import pandas as pd

embalses = spark.read.parquet("hdfs://namenode:9000/proyecto_xm/curated/PorcVoluUtilDiar")
embalses.printSchema()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", None)

muestra_embalses = (
    embalses.select("metrica", "fecha", "tipo_entidad", "entidad", "valor", "archivo_origen", "year")
    .orderBy("fecha", "entidad")
    .limit(20)
    .toPandas()
)

muestra_embalses["valor"] = (muestra_embalses["valor"] * 100).round(2)

display(muestra_embalses)

root
 |-- metrica: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora: integer (nullable = true)
 |-- tipo_entidad: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- year: integer (nullable = true)



,metrica,fecha,tipo_entidad,entidad,valor,archivo_origen,year
0,PorcVoluUtilDiar,2020-01-01,Embalse,AGREGADO BOGOTA,45.35,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
1,PorcVoluUtilDiar,2020-01-01,Embalse,ALTOANCHICAYA,74.89,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
2,PorcVoluUtilDiar,2020-01-01,Embalse,AMANI,94.55,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
3,PorcVoluUtilDiar,2020-01-01,Embalse,BETANIA,73.79,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
4,PorcVoluUtilDiar,2020-01-01,Embalse,CALIMA1,81.36,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
5,PorcVoluUtilDiar,2020-01-01,Embalse,CHUZA,69.20,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
6,PorcVoluUtilDiar,2020-01-01,Embalse,EL QUIMBO,69.16,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
7,PorcVoluUtilDiar,2020-01-01,Embalse,ESMERALDA,12.26,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
8,PorcVoluUtilDiar,2020-01-01,Embalse,GUAVIO,86.60,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020
9,PorcVoluUtilDiar,2020-01-01,Embalse,MIRAFLORES,77.01,hdfs://namenode:9000/proyecto_xm/raw/xm/PorcVoluUtilDiar/year=2020/PorcVoluUtilDiar_2020-01-01_2020-01-30.json,2020


Antes de analizar los niveles de almacenamiento, se verifica la cobertura temporal, el volumen de registros, la cantidad de entidades y el rango de valores disponibles en la métrica `PorcVoluUtilDiar`.

In [29]:
embalses.createOrReplaceTempView("embalses")

perfil_embalses = spark.sql("""
SELECT MIN(fecha) AS fecha_min, MAX(fecha) AS fecha_max,
       COUNT(*) AS registros, COUNT(DISTINCT entidad) AS Embalses,
       ROUND(MIN(valor)*100, 2) AS Porcentaje_minimo,
       ROUND(MAX(valor)*100, 2) AS Porcentaje_maximo
FROM embalses
""").toPandas()

display(perfil_embalses)

,fecha_min,fecha_max,registros,Embalses,Porcentaje_minimo,Porcentaje_maximo
0,2020-01-01,2026-11-19,58946,24,0.03,142.05
